In [7]:
# ============================================================
# Data Quality Testing - AI Adoption Pipeline
# ============================================================
# Part 1: Custom SQL Tests
# Part 2: Great Expectations (showcase)
# ============================================================

import os
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv

load_dotenv()

client = bigquery.Client.from_service_account_json(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS"),
    project=os.getenv("BQ_PROJECT_ID")
)

PROJECT   = os.getenv("BQ_PROJECT_ID")
DATASET   = os.getenv("BQ_DW_DATASET")   # ai_adoption_dw

def run_test(test_name, query):
    """Run a SQL test and print PASS or FAIL."""
    df = client.query(query).to_dataframe()
    failed_rows = len(df)
    status = "✅ PASS" if failed_rows == 0 else f"❌ FAIL ({failed_rows} rows)"
    print(f"{status} | {test_name}")
    if failed_rows > 0:
        display(df.head(5))
    return failed_rows

In [8]:
# ============================================================
# TEST 1: Null Checks
# ============================================================

null_tests = {

    "fact_ai_survey: survey_key not null": f"""
        select survey_key from `{PROJECT}.{DATASET}.fact_ai_survey`
        where survey_key is null
    """,

    "fact_ai_survey: company_key not null": f"""
        select company_key from `{PROJECT}.{DATASET}.fact_ai_survey`
        where company_key is null
    """,

    "fact_ai_survey: date_key not null": f"""
        select date_key from `{PROJECT}.{DATASET}.fact_ai_survey`
        where date_key is null
    """,

    "dim_company: company_id not null": f"""
        select company_key from `{PROJECT}.{DATASET}.dim_company`
        where company_id is null
    """,

    "dim_date: survey_year not null": f"""
        select date_key from `{PROJECT}.{DATASET}.dim_date`
        where survey_year is null
    """,

    "dim_ai_tool: ai_primary_tool not null": f"""
        select ai_tool_key from `{PROJECT}.{DATASET}.dim_ai_tool`
        where ai_primary_tool is null
    """,
}

print("=" * 50)
print("TEST 1: NULL CHECKS")
print("=" * 50)
for test_name, query in null_tests.items():
    run_test(test_name, query)

TEST 1: NULL CHECKS
✅ PASS | fact_ai_survey: survey_key not null
✅ PASS | fact_ai_survey: company_key not null
✅ PASS | fact_ai_survey: date_key not null
✅ PASS | dim_company: company_id not null
✅ PASS | dim_date: survey_year not null
✅ PASS | dim_ai_tool: ai_primary_tool not null


In [9]:
# ============================================================
# TEST 2: Duplicate Checks
# ============================================================

duplicate_tests = {

    "fact_ai_survey: survey_key is unique": f"""
        select survey_key, count(*) as cnt
        from `{PROJECT}.{DATASET}.fact_ai_survey`
        group by survey_key
        having cnt > 1
    """,

    "dim_company: company_key is unique": f"""
        select company_key, count(*) as cnt
        from `{PROJECT}.{DATASET}.dim_company`
        group by company_key
        having cnt > 1
    """,

    "dim_date: date_key is unique": f"""
        select date_key, count(*) as cnt
        from `{PROJECT}.{DATASET}.dim_date`
        group by date_key
        having cnt > 1
    """,

    "dim_ai_tool: ai_tool_key is unique": f"""
        select ai_tool_key, count(*) as cnt
        from `{PROJECT}.{DATASET}.dim_ai_tool`
        group by ai_tool_key
        having cnt > 1
    """,

    "dim_ai_adoption_stage: stage_key is unique": f"""
        select stage_key, count(*) as cnt
        from `{PROJECT}.{DATASET}.dim_ai_adoption_stage`
        group by stage_key
        having cnt > 1
    """,
}

print("=" * 50)
print("TEST 2: DUPLICATE CHECKS")
print("=" * 50)
for test_name, query in duplicate_tests.items():
    run_test(test_name, query)

TEST 2: DUPLICATE CHECKS
✅ PASS | fact_ai_survey: survey_key is unique
✅ PASS | dim_company: company_key is unique
✅ PASS | dim_date: date_key is unique
✅ PASS | dim_ai_tool: ai_tool_key is unique
✅ PASS | dim_ai_adoption_stage: stage_key is unique


In [11]:
# ============================================================
# TEST 3: Referential Integrity
# ============================================================

fk_tests = {

    "fact → dim_company: no orphan company_key": f"""
        select f.company_key
        from `{PROJECT}.{DATASET}.fact_ai_survey` f
        left join `{PROJECT}.{DATASET}.dim_company` d
            on f.company_key = d.company_key
        where d.company_key is null
    """,

    "fact → dim_date: no orphan date_key": f"""
        select f.date_key
        from `{PROJECT}.{DATASET}.fact_ai_survey` f
        left join `{PROJECT}.{DATASET}.dim_date` d
            on f.date_key = d.date_key
        where d.date_key is null
    """,

    "fact → dim_ai_tool: no orphan ai_tool_key": f"""
        select f.ai_tool_key
        from `{PROJECT}.{DATASET}.fact_ai_survey` f
        left join `{PROJECT}.{DATASET}.dim_ai_tool` d
            on f.ai_tool_key = d.ai_tool_key
        where d.ai_tool_key is null
    """,

    "fact → dim_ai_usecase: no orphan usecase_key": f"""
        select f.usecase_key
        from `{PROJECT}.{DATASET}.fact_ai_survey` f
        left join `{PROJECT}.{DATASET}.dim_ai_usecase` d
            on f.usecase_key = d.usecase_key
        where d.usecase_key is null
    """,

    "fact → dim_ai_adoption_stage: no orphan stage_key": f"""
        select f.stage_key
        from `{PROJECT}.{DATASET}.fact_ai_survey` f
        left join `{PROJECT}.{DATASET}.dim_ai_adoption_stage` d
            on f.stage_key = d.stage_key
        where d.stage_key is null
    """,

    "fact → dim_survey_source: no orphan source_key": f"""
        select f.source_key
        from `{PROJECT}.{DATASET}.fact_ai_survey` f
        left join `{PROJECT}.{DATASET}.dim_survey_source` d
            on f.source_key = d.source_key
        where d.source_key is null
    """,
}

print("=" * 50)
print("TEST 3: REFERENTIAL INTEGRITY")
print("=" * 50)
for test_name, query in fk_tests.items():
    run_test(test_name, query)

TEST 3: REFERENTIAL INTEGRITY
✅ PASS | fact → dim_company: no orphan company_key
✅ PASS | fact → dim_date: no orphan date_key
✅ PASS | fact → dim_ai_tool: no orphan ai_tool_key
✅ PASS | fact → dim_ai_usecase: no orphan usecase_key
✅ PASS | fact → dim_ai_adoption_stage: no orphan stage_key
✅ PASS | fact → dim_survey_source: no orphan source_key


In [13]:

# Fixed code: Using your specific project and dataset directly in the query
query = """
SELECT 
    f.stage_key as fact_key, 
    d.stage_key as dim_key,
    COUNT(*) as record_count
FROM `ai-adoption-pipeline.ai_adoption_dw.fact_ai_survey` f
LEFT JOIN `ai-adoption-pipeline.ai_adoption_dw.dim_ai_adoption_stage` d 
  ON f.stage_key = d.stage_key
WHERE d.stage_key IS NULL
GROUP BY 1, 2
LIMIT 10
"""

# Execute the query
df_debug = client.query(query).to_dataframe()

# Display the findings
if df_debug.empty:
    print("Success! No orphan records found.")
else:
    print("Found orphan records. See fact_key values below:")
    display(df_debug)

Success! No orphan records found.


In [14]:
# ============================================================
# TEST 4: Business Logic
# ============================================================

logic_tests = {

    "task_automation_rate between 0 and 100": f"""
        select survey_key, task_automation_rate
        from `{PROJECT}.{DATASET}.fact_ai_survey`
        where task_automation_rate < 0
           or task_automation_rate > 100
    """,

    "revenue_growth_percent between -100 and 100": f"""
        select survey_key, revenue_growth_percent
        from `{PROJECT}.{DATASET}.fact_ai_survey`
        where revenue_growth_percent < -100
           or revenue_growth_percent > 100
    """,

    "jobs_displaced >= 0": f"""
        select survey_key, jobs_displaced
        from `{PROJECT}.{DATASET}.fact_ai_survey`
        where jobs_displaced < 0
    """,

    "jobs_created >= 0": f"""
        select survey_key, jobs_created
        from `{PROJECT}.{DATASET}.fact_ai_survey`
        where jobs_created < 0
    """,

    "survey_year is valid (2023-2026)": f"""
        select date_key, survey_year
        from `{PROJECT}.{DATASET}.dim_date`
        where survey_year not in (2023, 2024, 2025, 2026)
    """,

    "quarter is valid (Q1-Q4)": f"""
        select date_key, quarter
        from `{PROJECT}.{DATASET}.dim_date`
        where quarter not in ('Q1', 'Q2', 'Q3', 'Q4')
    """,

    "stage_order is valid (1-5)": f"""
        select stage_key, adoption_stage, stage_order
        from `{PROJECT}.{DATASET}.dim_ai_adoption_stage`
        where stage_order not between 1 and 5
    """,

    "net_jobs_change = jobs_created - jobs_displaced": f"""
        select survey_key, net_jobs_change,
               jobs_created - jobs_displaced as expected
        from `{PROJECT}.{DATASET}.fact_ai_survey`
        where net_jobs_change != jobs_created - jobs_displaced
    """,
}

print("=" * 50)
print("TEST 4: BUSINESS LOGIC")
print("=" * 50)
for test_name, query in logic_tests.items():
    run_test(test_name, query)

TEST 4: BUSINESS LOGIC
✅ PASS | task_automation_rate between 0 and 100
✅ PASS | revenue_growth_percent between -100 and 100
✅ PASS | jobs_displaced >= 0
✅ PASS | jobs_created >= 0
✅ PASS | survey_year is valid (2023-2026)
✅ PASS | quarter is valid (Q1-Q4)
✅ PASS | stage_order is valid (1-5)
✅ PASS | net_jobs_change = jobs_created - jobs_displaced


In [15]:
# ============================================================
# SUMMARY REPORT
# ============================================================

all_tests = {**null_tests, **duplicate_tests, **fk_tests, **logic_tests}

results = []
for test_name, query in all_tests.items():
    df = client.query(query).to_dataframe()
    failed = len(df)
    results.append({
        "test": test_name,
        "status": "PASS" if failed == 0 else "FAIL",
        "failed_rows": failed
    })

summary = pd.DataFrame(results)
total   = len(summary)
passed  = len(summary[summary["status"] == "PASS"])
failed  = len(summary[summary["status"] == "FAIL"])

print(f"\n{'=' * 50}")
print(f"SUMMARY: {passed}/{total} tests passed, {failed} failed")
print(f"{'=' * 50}")
display(summary)


SUMMARY: 25/25 tests passed, 0 failed


,test,status,failed_rows
0,fact_ai_survey: survey_key not null,PASS,0
1,fact_ai_survey: company_key not null,PASS,0
2,fact_ai_survey: date_key not null,PASS,0
3,dim_company: company_id not null,PASS,0
4,dim_date: survey_year not null,PASS,0
5,dim_ai_tool: ai_primary_tool not null,PASS,0
6,fact_ai_survey: survey_key is unique,PASS,0
7,dim_company: company_key is unique,PASS,0
8,dim_date: date_key is unique,PASS,0
9,dim_ai_tool: ai_tool_key is unique,PASS,0


In [16]:
# ============================================================
# PART 2: Great Expectations (Verified for v0.18.22)
# ============================================================

import great_expectations as ge
import pandas as pd
from google.cloud import bigquery

# 0. Initialize the BigQuery Client (Fixes the NameError)
client = bigquery.Client()

# 1. Load your fact table from BigQuery
query = f"select * from `ai-adoption-pipeline.ai_adoption_dw.fact_ai_survey` limit 10000"
df = client.query(query).to_dataframe()

# 2. Wrap the Pandas DataFrame into a GE Dataset
ge_df = ge.dataset.PandasDataset(df)

print("Running data quality checks...")

# 3. Define Expectations (The Rules)
# Rule 1: survey_key must not be null
results_1 = ge_df.expect_column_values_to_not_be_null("survey_key")

# Rule 2: task_automation_rate between 0 and 100
results_2 = ge_df.expect_column_values_to_be_between(
    column="task_automation_rate", min_value=0, max_value=100
)

# Rule 3: jobs_displaced must be 0 or higher
results_3 = ge_df.expect_column_values_to_be_between(
    column="jobs_displaced", min_value=0
)

# 4. Display a clean summary for your presentation
print("\n" + "="*50)
print("GREAT EXPECTATIONS: DATA QUALITY REPORT (v0.18.22)")
print("="*50)

all_checks = [
    ("Survey Key Integrity", results_1),
    ("Automation Rate Bounds", results_2),
    ("Jobs Displaced Logic", results_3)
]

for name, res in all_checks:
    status = "✅ PASS" if res.success else "❌ FAIL"
    print(f"{status} | {name}")

if all(res.success for name, res in all_checks):
    print("\n🎉 DATA PIPELINE VERIFIED: High Quality Data.")
else:
    print("\n⚠️ WARNING: Data quality issues detected.")

Running data quality checks...

GREAT EXPECTATIONS: DATA QUALITY REPORT (v0.18.22)
✅ PASS | Survey Key Integrity
✅ PASS | Automation Rate Bounds
✅ PASS | Jobs Displaced Logic

🎉 DATA PIPELINE VERIFIED: High Quality Data.
